In [8]:
import base64
import requests
import pandas as pd
import os

# OpenAI API Key
api_key = "YOUR_OPENAI_API_KEY"

In [4]:
df = pd.read_csv('../main.csv', index_col=0)

In [21]:
## JUST A QUIK TEST
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

def decode_image(base64_string, output_image_path):
    image_data = base64.b64decode(base64_string)
    with open(output_image_path, "wb") as image_file:
        image_file.write(image_data)

image_path = '/home/data/v.moskvoretskii/taxo_demo_images/images'
row_number = 0
item = df.loc[row_number]
idx = item['wordnet_id']

# Path to your image
path2image1 = f'{image_path}/{item["model_a"]}/{idx}.png'
if not f'{idx}.png' in os.listdir(f'{image_path}/{item["model_a"]}/'):
    path2image1 = f'{image_path}/{item["model_a"]}/{idx}.jpg'

    if not f'{idx}.jpg' in os.listdir(f'{image_path}/{item["model_a"]}/'):
        print('no such image')

# Path to your image
path2image2 = f'{image_path}/{item["model_b"]}/{idx}.png'
if not f'{idx}.png' in os.listdir(f'{image_path}/{item["model_b"]}/'):
    path2image2 = f'{image_path}/{item["model_b"]}/{idx}.jpg'
    if not f'{idx}.jpg' in os.listdir(f'{image_path}/{item["model_b"]}/'):
        print('no such image')


# Getting the base64 string
base64_image = encode_image(path2image1)
#decoding back
decode_image(base64_image, "output_image.jpg")
# make sure it is okay

# Getting the base64 string
base64_image2 = encode_image(path2image2)
#decoding back
decode_image(base64_image2, "output_image2.jpg")
# make sure it is okay

In [23]:
item['core_lemma']

'coin'

In [ ]:
preference_prompt = '''Please act as an impartial judge and evaluate the quality of the images provided by two
AI image generators to the user question displayed below. You should choose the assistant that
follows the user’s instructions and answers the user’s question better. Your evaluation
should consider factors such as the helpfulness, relevance, accuracy, depth, creativity,
and level of detail of their responses. Begin your evaluation by comparing the two
responses and provide a short explanation. Avoid any position biases and ensure that the
order in which the responses were presented does not influence your decision. Do not allow
the length of the responses to influence your evaluation. Do not favor certain names of
the assistants. Be as objective as possible. After providing your explanation, output your
final verdict by strictly following this format: "[[A]]" if assistant A is better, "[[B]]"
if assistant B is better, and "[[C]]" for a tie.'''

headers = {
  "Content-Type": "application/json",
  "Authorization": f"Bearer {api_key}"
}

payload = {
  "model": "gpt-4o-mini",
  "messages": [
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": f'{preference_prompt} \n [User Prompt] \n {item["core_lemma"]} [Start of first image]'
        },
        {
          "type": "image_url",
          "image_url": {
            "url": f"data:image/jpeg;base64,{base64_image_a}"
          }
        },
        {
          "type": "text",
          "text": '[End of first image] \n [Start of second image]'
        },
        {
          "type": "image_url",
          "image_url": {
            "url": f"data:image/jpeg;base64,{base64_image_b}"
          }
        },
        {
          "type": "text",
          "text": '[End of second image]'
        },
      ]
    }
  ],
  "max_tokens": 300
}

response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)

print(response.json())